We solve the binary detection problem:

- $\mathcal{H}_0: y[n]=w[n],\; w[n]\sim\mathcal{CN}(0,\sigma_w^2)$
- $\mathcal{H}_1: y[n]=s[n]+w[n],\; s[n]\sim\mathcal{CN}(0,\gamma\sigma_w^2)$

with test statistic

$$
U=\frac{1}{\sigma_{w,\text{assumed}}^2}\sum_{n=1}^{N}|y[n]|^2.
$$

Decision rule:

$$
\text{Decide }\mathcal{H}_1\;\text{if}\;U>\eta,\quad\text{else }\mathcal{H}_0.
$$

NP threshold for false alarm $\alpha$:

$$
\eta_\alpha = \frac12 F^{-1}_{\chi^2_{2N}}(1-\alpha).
$$

Theoretical detection (Gaussian random signal model):

$$
P_d^{\text{th}}(\gamma;\alpha)=1-F_{\chi^2_{2N}}\!\left(\frac{2\eta_\alpha}{1+\gamma}\right).
$$

Primary target: $P_d$ at $\alpha=0.01$ over SNR grid $\{-20,-18,\dots,0\}$ dB, plus empirical agreement check.

Key formulas used in implementation:

- SNR conversion: $\gamma = 10^{\text{SNR}_{dB}/10}$.
- Complex AWGN generation (power-correct):
  $$
  w = \frac{\sigma}{\sqrt{2}}\left(\mathcal{N}(0,1)+j\mathcal{N}(0,1)\right),\quad \sigma^2=\sigma_w^2.
  $$
- NP threshold for each $P_{fa}=\alpha_m\in\{0.01,0.05,0.1\}$:
  $$
  \eta(\alpha_m)=\frac12\chi^2_{2N}^{-1}(1-\alpha_m).
  $$
- Theoretical NP detection:
  $$
  P_d^{\text{th}}(\gamma;\alpha_m)=1-F_{\chi^2_{2N}}\!\left(\frac{2\eta(\alpha_m)}{1+\gamma}\right).
  $$
- MC estimators:
  $$
  \widehat{P}_{fa}=\frac{1}{T}\sum_{t=1}^T \mathbf{1}\{U_t^{(0)}>\eta\},\qquad
  \widehat{P}_{d}=\frac{1}{T}\sum_{t=1}^T \mathbf{1}\{U_t^{(1)}>\eta\}.
  $$
- CFAR baseline threshold:
  $$
  \eta_{\text{CFAR}}=\text{Quantile}_{1-\alpha}\big(\{U_t^{(0)}\}_{t=1}^{T_{\text{cal}}}\big).
  $$
- Cyclostationary baseline statistic:
  $$
  \hat{R}_y^\alpha(\tau)=\frac{1}{N-\tau}\sum_{n=1}^{N-\tau}y[n+\tau]y^*[n]e^{-j2\pi\alpha n},\quad
  T_{\text{CFD}}=\max_{\alpha,\tau}\left|\hat{R}_y^\alpha(\tau)\right|.
  $$
- ROC/AUC:
  $$
  \text{AUC}=\int_0^1 P_d(P_{fa})\,dP_{fa}\approx \text{trapz}(P_d,P_{fa}).
  $$

In [ ]:
import time
import json
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import chi2
from scipy import signal
from scipy import optimize
from scipy import linalg

Next we define data-generation utilities for hypothesis samples and energy statistics.

- **Inputs:** $N$, SNR in dB, trial count, assumed noise power $\sigma_{w,\text{assumed}}^2$, mismatch $\delta$ dB.
- **Outputs:** $y^{(0)}$, $y^{(1)}$, $\sigma_{w,\text{true}}^2$, $\gamma$, and energy statistics $U$.
- Key equations:
  - $\sigma_{w,\text{true}}^2=\sigma_{w,\text{assumed}}^2\cdot 10^{\delta/10}$
  - $w=\sqrt{\sigma_{w,\text{true}}^2/2}(a+jb)$
  - $U=\sigma_{w,\text{assumed}}^{-2}\sum_n |y[n]|^2$

In [ ]:
def db_to_linear(db):
    return 10.0 ** (db / 10.0)

def complex_awgn(num_intervals, N, sigma2, rng):
    sigma = np.sqrt(sigma2)
    noise = (sigma / np.sqrt(2.0)) * (
        rng.standard_normal((num_intervals, N)) + 1j * rng.standard_normal((num_intervals, N))
    )
    return noise

def generate_hypothesis_samples(snr_db, N, num_intervals, sigma_w2_assumed, mismatch_db, rng):
    gamma = db_to_linear(snr_db)
    sigma_w2_true = sigma_w2_assumed * db_to_linear(mismatch_db)

    y0 = complex_awgn(num_intervals, N, sigma_w2_true, rng)
    w1 = complex_awgn(num_intervals, N, sigma_w2_true, rng)
    s = (np.sqrt(gamma * sigma_w2_true) / np.sqrt(2.0)) * (
        rng.standard_normal((num_intervals, N)) + 1j * rng.standard_normal((num_intervals, N))
    )
    y1 = s + w1
    return y0, y1, sigma_w2_true, gamma

def energy_statistic(y, sigma_w2_assumed):
    return np.sum(np.abs(y) ** 2, axis=1) / sigma_w2_assumed

def simulate_energy_statistics_batched(
    snr_db, N, num_trials, batch_size, sigma_w2_assumed, mismatch_db, rng
):
    U0_parts, U1_parts = [], []
    generated = 0
    while generated < num_trials:
        cur = min(batch_size, num_trials - generated)
        y0, y1, _, _ = generate_hypothesis_samples(
            snr_db=snr_db,
            N=N,
            num_intervals=cur,
            sigma_w2_assumed=sigma_w2_assumed,
            mismatch_db=mismatch_db,
            rng=rng,
        )
        U0_parts.append(energy_statistic(y0, sigma_w2_assumed))
        U1_parts.append(energy_statistic(y1, sigma_w2_assumed))
        generated += cur
    return np.concatenate(U0_parts), np.concatenate(U1_parts)

The next block implements the algorithm core with formula-tagged steps from the design contract.

- **Computes:** NP thresholds, theoretical $P_d$, empirical $P_d/P_{fa}$, ROC/AUC, cyclostationary statistic, complexity/runtime helpers.
- **Inputs:** generated statistics $U$, SNR grid, $P_{fa}$ points.
- **Outputs:** callable primitives used by the sweep.
- Key equations:
  - $\eta(\alpha)=\frac12\chi^{-1}_{2N}(1-\alpha)$
  - $P_d^{th}=1-F_{\chi^2_{2N}}(2\eta/(1+\gamma))$
  - $d=\mathbf{1}\{U>\eta\}$

In [ ]:
# Step 1 formula: N=256, alpha=0.01, G_dB, gamma_i=10^(G_dB/10), T=100000
def initialize_constants_and_grids(snr_db_grid, N, alpha, num_trials):
    gamma_grid = db_to_linear(np.array(snr_db_grid, dtype=float))
    return int(N), float(alpha), gamma_grid, int(num_trials)

# Step 2 formula: seed=2026, sigma_w2_assumed>0, mismatch grid
def setup_rng_and_noise_reference(seed, sigma_w2_assumed, mismatch_grid_db):
    rng = np.random.default_rng(seed)
    return rng, float(sigma_w2_assumed), np.array(mismatch_grid_db, dtype=float)

# Step 3 formula: eta(alpha_m)=0.5*chi2.ppf(1-alpha_m,2N)
def compute_np_thresholds(N, pfa_points):
    eta_map = {}
    for alpha_m in pfa_points:
        eta_map[float(alpha_m)] = 0.5 * chi2.ppf(1.0 - float(alpha_m), df=2 * N)
    return eta_map

# Step 4 formula: Pd_th=1-chi2.cdf(2*eta/(1+gamma),2N)
def compute_theoretical_pd_curve(gamma_grid, eta, N):
    x = (2.0 * eta) / (1.0 + gamma_grid)
    return 1.0 - chi2.cdf(x, df=2 * N)

# Step 8 + 9 formulas: decisions and empirical estimates
def empirical_detection_probabilities(U0, U1, eta):
    d0 = (U0 > eta).astype(np.float64)
    d1 = (U1 > eta).astype(np.float64)
    pfa_emp = float(np.mean(d0))
    pd_emp = float(np.mean(d1))
    return pfa_emp, pd_emp

# Cyclostationary baseline statistic:
# T_CFD=max_{alpha,tau}|R_hat_y^alpha(tau)|, R_hat_y^alpha(tau)=1/(N-tau)*sum y[n+tau]y*[n]e^{-j2pi alpha n}
def cyclostationary_statistic_batch(y, alpha_grid, tau_grid):
    num_intervals, N = y.shape
    stat = np.zeros(num_intervals, dtype=float)
    for tau in tau_grid:
        if tau >= N:
            continue
        prod = y[:, tau:] * np.conj(y[:, :-tau])
        n = np.arange(N - tau)
        for alpha in alpha_grid:
            phase = np.exp(-1j * 2.0 * np.pi * alpha * n)
            r_hat = np.mean(prod * phase[None, :], axis=1)
            stat = np.maximum(stat, np.abs(r_hat))
    return stat

def simulate_cfd_statistics_batched(
    snr_db, N, num_trials, batch_size, sigma_w2_assumed, mismatch_db, alpha_grid, tau_grid, rng
):
    T0_parts, T1_parts = [], []
    generated = 0
    while generated < num_trials:
        cur = min(batch_size, num_trials - generated)
        y0, y1, _, _ = generate_hypothesis_samples(
            snr_db, N, cur, sigma_w2_assumed, mismatch_db, rng
        )
        T0_parts.append(cyclostationary_statistic_batch(y0, alpha_grid, tau_grid))
        T1_parts.append(cyclostationary_statistic_batch(y1, alpha_grid, tau_grid))
        generated += cur
    return np.concatenate(T0_parts), np.concatenate(T1_parts)

# Step 10 formula: ROC threshold sweep and AUC
def compute_roc_auc(U0, U1, num_thresholds=120):
    thresholds = np.quantile(U0, np.linspace(0.0, 1.0, num_thresholds))
    pfa_curve = np.array([np.mean(U0 > th) for th in thresholds], dtype=float)
    pd_curve = np.array([np.mean(U1 > th) for th in thresholds], dtype=float)
    order = np.argsort(pfa_curve)
    pfa_sorted = pfa_curve[order]
    pd_sorted = pd_curve[order]
    auc = float(np.trapz(pd_sorted, pfa_sorted))
    return pfa_sorted, pd_sorted, auc

# Step 11 formula: runtime_ms_per_interval=1000*(t_end-t_start)/N_intervals
def runtime_per_interval_ms(elapsed_sec, num_intervals_processed):
    return 1000.0 * elapsed_sec / max(num_intervals_processed, 1)

def complexity_ops_per_interval_estimate(N):
    # energy detector: |y|^2 (mult+add) and accumulation ~ O(N)
    return int(6 * N)

Next we build the evaluation sweep function `run_evaluation(eval_config)`.

- **Computes:** primary SNR sweep, all method curves per operating point, ROC/AUC, noise-uncertainty robustness, and snapshot sensitivity.
- **Inputs:** `eval_config` (SNR points, trials, thresholds, mismatch, snapshot points).
- **Outputs:** JSON-serializable `performance_data` and `evaluation_summary`.
- Key equations:
  - $\widehat{P}_{d}(i)=\frac1T\sum_t \mathbf{1}\{U_t^{(1,i)}>\eta\}$
  - $\Delta_{pd}(i)=|\widehat{P}_{d}(i)-P_d^{th}(i)|$
  - $\eta_{\text{CFAR}}=\text{Quantile}_{1-\alpha}(U_0^{\text{cal}})$

In [ ]:
def run_evaluation(eval_config):
    # Read operating points
    snr_points = list(eval_config["snr_points"])
    N = int(eval_config["N"])
    alpha_target = float(eval_config["alpha_target"])
    pfa_points = list(eval_config["pfa_points"])
    sigma_w2_assumed = float(eval_config["sigma_w2_assumed"])
    num_trials_energy = int(eval_config["num_trials_energy"])
    num_trials_cfar_cal = int(eval_config["num_trials_cfar_cal"])
    num_trials_cfd = int(eval_config["num_trials_cfd"])
    batch_size_energy = int(eval_config["batch_size_energy"])
    batch_size_cfd = int(eval_config["batch_size_cfd"])
    mismatch_grid_db = list(eval_config["noise_mismatch_db_list"])
    roc_snr_points = list(eval_config["roc_snr_points"])
    snapshot_points = list(eval_config["snapshot_points"])
    snapshot_snr_db = float(eval_config["snapshot_snr_db"])
    rng = np.random.default_rng(int(eval_config["random_seed"]))

    # Step 1
    N, alpha_target, gamma_grid, _ = initialize_constants_and_grids(
        snr_points, N, alpha_target, num_trials_energy
    )
    # Step 2
    rng, sigma_w2_assumed, mismatch_arr = setup_rng_and_noise_reference(
        int(eval_config["random_seed"]), sigma_w2_assumed, mismatch_grid_db
    )
    # Step 3
    eta_map = compute_np_thresholds(N, pfa_points)
    eta_np = eta_map[alpha_target]

    # Step 4 theoretical curves for fixed pfa points
    pd_theory_map = {}
    for pfa in pfa_points:
        pd_theory_map[str(pfa)] = compute_theoretical_pd_curve(gamma_grid, eta_map[pfa], N).tolist()

    # Storage
    pd_proposed = []
    pd_np_analytic_emp = []
    pd_cfar_emp = []
    pd_np_optimal = []
    pd_cfd_emp = []

    pfa_np_analytic_emp = []
    pfa_cfar_emp = []
    pfa_cfd_emp = []
    pd_gap_np = []

    roc_curves = {}
    auc_values = {}

    runtime_start = time.perf_counter()
    intervals_processed_for_runtime = 0

    # Primary sweep over EVERY operating point
    for snr_db in snr_points:
        gamma = db_to_linear(snr_db)

        # Step 5 + 6 + 7 for NP analytic/proposed data
        U0_np, U1_np = simulate_energy_statistics_batched(
            snr_db=snr_db,
            N=N,
            num_trials=num_trials_energy,
            batch_size=batch_size_energy,
            sigma_w2_assumed=sigma_w2_assumed,
            mismatch_db=0.0,
            rng=rng,
        )
        intervals_processed_for_runtime += 2 * num_trials_energy

        # Step 8 + 9 proposed/theory and NP analytic baseline
        pd_th = float(1.0 - chi2.cdf((2.0 * eta_np) / (1.0 + gamma), df=2 * N))
        pfa_np, pd_np = empirical_detection_probabilities(U0_np, U1_np, eta_np)

        pd_proposed.append(pd_th)                 # proposed: closed-form NP
        pd_np_analytic_emp.append(pd_np)          # baseline 1: analytic threshold MC
        pfa_np_analytic_emp.append(pfa_np)

        # Baseline 2: MC-calibrated CFAR threshold (independent calibration + test data)
        U0_cfar_cal, _ = simulate_energy_statistics_batched(
            snr_db=snr_db,
            N=N,
            num_trials=num_trials_cfar_cal,
            batch_size=batch_size_energy,
            sigma_w2_assumed=sigma_w2_assumed,
            mismatch_db=0.0,
            rng=rng,
        )
        eta_cfar = float(np.quantile(U0_cfar_cal, 1.0 - alpha_target))
        U0_cfar_test, U1_cfar_test = simulate_energy_statistics_batched(
            snr_db=snr_db,
            N=N,
            num_trials=num_trials_energy,
            batch_size=batch_size_energy,
            sigma_w2_assumed=sigma_w2_assumed,
            mismatch_db=0.0,
            rng=rng,
        )
        pfa_cfar, pd_cfar = empirical_detection_probabilities(U0_cfar_test, U1_cfar_test, eta_cfar)
        pd_cfar_emp.append(pd_cfar)
        pfa_cfar_emp.append(pfa_cfar)

        # Baseline 3: NP optimality benchmark (same closed-form optimum for this statistic family)
        pd_opt = float(1.0 - chi2.cdf((2.0 * eta_np) / (1.0 + gamma), df=2 * N))
        pd_np_optimal.append(pd_opt)

        # Baseline 4: Cyclostationary feature detector (independent calibration + test)
        alpha_grid = eval_config["cfd_alpha_grid"]
        tau_grid = eval_config["cfd_tau_grid"]
        T0_cfd_cal, _ = simulate_cfd_statistics_batched(
            snr_db=snr_db,
            N=N,
            num_trials=num_trials_cfd,
            batch_size=batch_size_cfd,
            sigma_w2_assumed=sigma_w2_assumed,
            mismatch_db=0.0,
            alpha_grid=alpha_grid,
            tau_grid=tau_grid,
            rng=rng,
        )
        eta_cfd = float(np.quantile(T0_cfd_cal, 1.0 - alpha_target))
        T0_cfd_test, T1_cfd_test = simulate_cfd_statistics_batched(
            snr_db=snr_db,
            N=N,
            num_trials=num_trials_cfd,
            batch_size=batch_size_cfd,
            sigma_w2_assumed=sigma_w2_assumed,
            mismatch_db=0.0,
            alpha_grid=alpha_grid,
            tau_grid=tau_grid,
            rng=rng,
        )
        pfa_cfd = float(np.mean(T0_cfd_test > eta_cfd))
        pd_cfd = float(np.mean(T1_cfd_test > eta_cfd))
        pfa_cfd_emp.append(pfa_cfd)
        pd_cfd_emp.append(pd_cfd)

        # Theory-simulation gap at alpha=0.01
        pd_gap_np.append(abs(pd_np - pd_th))

        # Step 10 ROC/AUC at selected SNR points (ideal mismatch)
        if snr_db in roc_snr_points:
            pfa_curve, pd_curve, auc = compute_roc_auc(
                U0_np, U1_np, num_thresholds=int(eval_config["roc_num_thresholds"])
            )
            roc_curves[str(snr_db)] = {
                "pfa": pfa_curve.tolist(),
                "pd": pd_curve.tolist(),
            }
            auc_values[str(snr_db)] = auc

    runtime_end = time.perf_counter()
    runtime_ms_interval = runtime_per_interval_ms(runtime_end - runtime_start, intervals_processed_for_runtime)

    # Noise uncertainty sensitivity (secondary check)
    mismatch_pd = {}
    mismatch_pfa = {}
    for mdb in mismatch_arr:
        U0_m, U1_m = simulate_energy_statistics_batched(
            snr_db=eval_config["noise_sensitivity_snr_db"],
            N=N,
            num_trials=num_trials_energy,
            batch_size=batch_size_energy,
            sigma_w2_assumed=sigma_w2_assumed,
            mismatch_db=float(mdb),
            rng=rng,
        )
        pfa_m, pd_m = empirical_detection_probabilities(U0_m, U1_m, eta_np)
        mismatch_pd[str(float(mdb))] = pd_m
        mismatch_pfa[str(float(mdb))] = pfa_m

    # Required additional sensitivity: snapshots (N) sensitivity
    snap_pd_proposed = []
    snap_pd_np = []
    snap_pd_cfar = []
    snap_pd_cfd = []
    for N_snap in snapshot_points:
        eta_np_snap = 0.5 * chi2.ppf(1.0 - alpha_target, df=2 * N_snap)
        gamma_snap = db_to_linear(snapshot_snr_db)

        pd_th_snap = float(1.0 - chi2.cdf((2.0 * eta_np_snap) / (1.0 + gamma_snap), df=2 * N_snap))
        snap_pd_proposed.append(pd_th_snap)

        U0_snap, U1_snap = simulate_energy_statistics_batched(
            snr_db=snapshot_snr_db,
            N=N_snap,
            num_trials=num_trials_energy,
            batch_size=batch_size_energy,
            sigma_w2_assumed=sigma_w2_assumed,
            mismatch_db=0.0,
            rng=rng,
        )
        _, pd_np_snap = empirical_detection_probabilities(U0_snap, U1_snap, eta_np_snap)
        snap_pd_np.append(pd_np_snap)

        U0_cal_snap, _ = simulate_energy_statistics_batched(
            snr_db=snapshot_snr_db,
            N=N_snap,
            num_trials=num_trials_cfar_cal,
            batch_size=batch_size_energy,
            sigma_w2_assumed=sigma_w2_assumed,
            mismatch_db=0.0,
            rng=rng,
        )
        eta_cfar_snap = float(np.quantile(U0_cal_snap, 1.0 - alpha_target))
        U0_test_snap, U1_test_snap = simulate_energy_statistics_batched(
            snr_db=snapshot_snr_db,
            N=N_snap,
            num_trials=num_trials_energy,
            batch_size=batch_size_energy,
            sigma_w2_assumed=sigma_w2_assumed,
            mismatch_db=0.0,
            rng=rng,
        )
        _, pd_cfar_snap = empirical_detection_probabilities(U0_test_snap, U1_test_snap, eta_cfar_snap)
        snap_pd_cfar.append(pd_cfar_snap)

        T0_cal_snap, _ = simulate_cfd_statistics_batched(
            snr_db=snapshot_snr_db,
            N=N_snap,
            num_trials=num_trials_cfd,
            batch_size=batch_size_cfd,
            sigma_w2_assumed=sigma_w2_assumed,
            mismatch_db=0.0,
            alpha_grid=eval_config["cfd_alpha_grid"],
            tau_grid=eval_config["cfd_tau_grid"],
            rng=rng,
        )
        eta_cfd_snap = float(np.quantile(T0_cal_snap, 1.0 - alpha_target))
        T0_test_snap, T1_test_snap = simulate_cfd_statistics_batched(
            snr_db=snapshot_snr_db,
            N=N_snap,
            num_trials=num_trials_cfd,
            batch_size=batch_size_cfd,
            sigma_w2_assumed=sigma_w2_assumed,
            mismatch_db=0.0,
            alpha_grid=eval_config["cfd_alpha_grid"],
            tau_grid=eval_config["cfd_tau_grid"],
            rng=rng,
        )
        snap_pd_cfd.append(float(np.mean(T1_test_snap > eta_cfd_snap)))

    # D9 guards
    method_arrays = {
        "proposed": pd_proposed,
        "np_analytic": pd_np_analytic_emp,
        "cfar_mc": pd_cfar_emp,
        "np_optimal": pd_np_optimal,
        "cyclostationary": pd_cfd_emp,
    }
    for method_name, arr in method_arrays.items():
        assert len(arr) == len(snr_points), f"{method_name}: result length mismatch"
        if len(set(round(v, 8) for v in arr)) == 1:
            print(f"WARNING: {method_name} produced constant results across all operating points — likely implementation bug.")
        for v in arr:
            assert 0.0 <= v <= 1.0, f"{method_name}: Pd out of [0,1]"

    for v in pfa_np_analytic_emp + pfa_cfar_emp + pfa_cfd_emp:
        assert 0.0 <= v <= 1.0, "Pfa out of [0,1]"

    performance_data = {
        "metric_curve": {
            "x": [float(x) for x in snr_points],
            "proposed": [float(v) for v in pd_proposed],
            "np_analytic": [float(v) for v in pd_np_analytic_emp],
            "cfar_mc": [float(v) for v in pd_cfar_emp],
            "np_optimal": [float(v) for v in pd_np_optimal],
            "cyclostationary": [float(v) for v in pd_cfd_emp],
            "x_label": "SNR (dB)",
            "y_label": "Detection Probability Pd @ Pfa=0.01"
        },
        "pfa_curve": {
            "x": [float(x) for x in snr_points],
            "np_analytic": [float(v) for v in pfa_np_analytic_emp],
            "cfar_mc": [float(v) for v in pfa_cfar_emp],
            "cyclostationary": [float(v) for v in pfa_cfd_emp],
            "x_label": "SNR (dB)",
            "y_label": "Empirical Pfa"
        },
        "theory_empirical_gap": {
            "x": [float(x) for x in snr_points],
            "np_gap_abs": [float(v) for v in pd_gap_np],
            "x_label": "SNR (dB)"
        },
        "roc_curves": roc_curves,
        "roc_auc": {k: float(v) for k, v in auc_values.items()},
        "noise_uncertainty_sensitivity": {
            "x": [float(x) for x in mismatch_arr.tolist()],
            "pd_np_analytic": [float(mismatch_pd[str(float(x))]) for x in mismatch_arr.tolist()],
            "pfa_np_analytic": [float(mismatch_pfa[str(float(x))]) for x in mismatch_arr.tolist()],
            "x_label": "Noise power mismatch (dB)"
        },
        "snapshot_sensitivity": {
            "x": [int(v) for v in snapshot_points],
            "proposed": [float(v) for v in snap_pd_proposed],
            "np_analytic": [float(v) for v in snap_pd_np],
            "cfar_mc": [float(v) for v in snap_pd_cfar],
            "cyclostationary": [float(v) for v in snap_pd_cfd],
            "x_label": "Number of samples N"
        },
        "runtime_ms_per_interval": float(runtime_ms_interval),
        "complexity_ops_per_interval": int(complexity_ops_per_interval_estimate(N)),
        "thresholds": {str(k): float(v) for k, v in eta_map.items()},
        "pd_theory_map": pd_theory_map
    }

    evaluation_summary = {
        "pfa_constraint_target": alpha_target,
        "max_abs_theory_empirical_pd_gap": float(np.max(np.array(pd_gap_np))),
        "pd_at_minus10db_theory": float(pd_proposed[snr_points.index(-10)]),
        "pd_at_minus10db_empirical_np": float(pd_np_analytic_emp[snr_points.index(-10)]),
    }

    return performance_data, evaluation_summary

# AUTO-WISP RESULTS HELPER
def _autowisp_normalize_results(candidate):
    if candidate is None:
        candidate = {}
    if not isinstance(candidate, dict):
        candidate = {'performance_data': {'raw_results': candidate}}
    performance_data = candidate.get('performance_data')
    if not isinstance(performance_data, dict):
        for key in ('metrics', 'results', 'evaluation_results', 'raw_results'):
            value = candidate.get(key)
            if isinstance(value, dict):
                performance_data = value
                break
    if not isinstance(performance_data, dict):
        performance_data = {'raw_results': performance_data if performance_data is not None else {}}
    report_assets = candidate.get('report_assets')
    if not isinstance(report_assets, dict):
        report_assets = {}
    report_assets.setdefault('problem_summary', 'Recovered from notebook auto-debug path.')
    report_assets.setdefault('solution_summary', 'Execution contract was normalized to ensure RESULTS generation.')
    report_assets.setdefault('evaluation_summary', 'Primary metric exported as P_d_at_P_fa_0_01.')
    performance_data = _autowisp_to_builtin(performance_data)
    tables = report_assets.get('tables')
    if not isinstance(tables, list) or not tables:
        tables = _autowisp_build_tables(performance_data)
    report_assets['tables'] = tables
    figures = report_assets.get('figures') or report_assets.get('figures_metadata') or []
    if not isinstance(figures, list):
        figures = []
    if not figures:
        figures = _autowisp_build_figures(performance_data)
    report_assets['figures'] = figures
    report_assets['figures_metadata'] = figures
    candidate['algorithm'] = candidate.get('algorithm') or 'Notebook Auto-Debug'
    candidate['elapsed_sec'] = float(candidate.get('elapsed_sec') or 0.0)
    candidate['performance_data'] = performance_data
    candidate['report_assets'] = report_assets
    return candidate

def _autowisp_to_builtin(value):
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, dict):
        return {str(k): _autowisp_to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [_autowisp_to_builtin(v) for v in value]
    if hasattr(value, 'tolist'):
        return _autowisp_to_builtin(value.tolist())
    if hasattr(value, 'item'):
        try:
            return value.item()
        except Exception:
            pass
    return str(value)

def _autowisp_curve_specs(perf_data):
    if not isinstance(perf_data, dict):
        return []
    _x_keys = ('x', 'variable_points', 'operating_points', 'snr', 'snr_points', 'snr_db')
    top_x = None
    for key in _x_keys:
        value = perf_data.get(key)
        if isinstance(value, list) and len(value) >= 2:
            top_x = value
            break
    specs = []
    for metric_name, payload in perf_data.items():
        if not isinstance(payload, dict):
            continue
        x_values = None
        for key in _x_keys:
            value = payload.get(key)
            if isinstance(value, list) and len(value) >= 2:
                x_values = value
                break
        if x_values is None:
            x_values = top_x
        if not isinstance(x_values, list) or len(x_values) < 2:
            continue
        _skip_keys = set(_x_keys) | {'x_label', 'xlabel', 'ylabel', 'title'}
        series = {}
        for series_name, series_values in payload.items():
            if series_name in _skip_keys:
                continue
            if isinstance(series_values, list) and len(series_values) == len(x_values):
                series[str(series_name)] = series_values
        if series:
            specs.append({
                'metric': str(metric_name),
                'x': x_values,
                'x_label': payload.get('x_label') or payload.get('xlabel') or 'Operating Point',
                'series': series,
            })
    return specs

def _autowisp_build_tables(perf_data):
    tables = []
    for spec in _autowisp_curve_specs(perf_data):
        columns = [spec['x_label']] + [name.replace('_', ' ').title() for name in spec['series'].keys()]
        rows = []
        for idx, x_value in enumerate(spec['x']):
            rows.append([x_value] + [values[idx] for values in spec['series'].values()])
        tables.append({
            'title': spec['metric'].replace('_', ' ').title(),
            'columns': columns,
            'rows': rows,
        })
    return tables

def _autowisp_build_figures(perf_data):
    figures = []
    for spec in _autowisp_curve_specs(perf_data):
        figures.append({
            'title': spec['metric'].replace('_', ' ').title(),
            'xlabel': spec['x_label'],
            'ylabel': spec['metric'].replace('_', ' '),
            'x': spec['x'],
            'series': spec['series'],
        })
    return figures

if 'run_experiment' not in globals():
    def run_experiment(eval_config=None):
        eval_config = dict(eval_config or globals().get('EVAL_CONFIG') or {})
        existing_runner = globals().get('run_evaluation')
        if callable(existing_runner):
            return _autowisp_normalize_results(existing_runner(eval_config))
        existing_results = globals().get('RESULTS')
        if isinstance(existing_results, dict):
            return _autowisp_normalize_results(existing_results)
        for key in ('performance_data', 'metrics', 'results', 'evaluation_results'):
            value = globals().get(key)
            if isinstance(value, dict):
                return _autowisp_normalize_results({'performance_data': value})
        raise RuntimeError('Notebook auto-debug could not infer performance_data for RESULTS generation')



Now we execute the full experiment with fixed hyperparameters and export the required `RESULTS` dictionary.

- **Computes:** one complete evaluation run.
- **Inputs:** named constants (SNR grid, trials, $N$, $P_{fa}$, mismatch set, ROC setup).
- **Outputs:** `RESULTS = {algorithm, elapsed_sec, performance_data, report_assets}`.
- Key equation references include $P_d^{th}$, $\widehat{P}_d$, and $\eta_\alpha$ computed inside the sweep.

In [ ]:
np.random.seed(42)

# Hyperparameters (named)
N_SAMPLES = 256
ALPHA_TARGET = 0.01
SNR_POINTS_DB = [-20, -18, -16, -14, -12, -10, -8, -6, -4, -2, 0]
PFA_POINTS = [0.01, 0.05, 0.1]
SIGMA_W2_ASSUMED = 1.0
RANDOM_SEED = 2026

NUM_TRIALS_ENERGY = 100000
NUM_TRIALS_CFAR_CAL = 60000
NUM_TRIALS_CFD = 3000
BATCH_SIZE_ENERGY = 5000
BATCH_SIZE_CFD = 300

NOISE_MISMATCH_DB_LIST = [-1.0, 0.0, 1.0]
NOISE_SENSITIVITY_SNR_DB = -10.0

ROC_SNR_POINTS = [-14, -10, -6]
ROC_NUM_THRESHOLDS = 120

SNAPSHOT_POINTS = [64, 128, 256, 512]
SNAPSHOT_SNR_DB = -10.0

CFD_ALPHA_GRID = [0.05, 0.10, 0.20]
CFD_TAU_GRID = [1, 2, 4]

eval_config = {
    "N": N_SAMPLES,
    "alpha_target": ALPHA_TARGET,
    "snr_points": SNR_POINTS_DB,
    "pfa_points": PFA_POINTS,
    "sigma_w2_assumed": SIGMA_W2_ASSUMED,
    "random_seed": RANDOM_SEED,
    "num_trials_energy": NUM_TRIALS_ENERGY,
    "num_trials_cfar_cal": NUM_TRIALS_CFAR_CAL,
    "num_trials_cfd": NUM_TRIALS_CFD,
    "batch_size_energy": BATCH_SIZE_ENERGY,
    "batch_size_cfd": BATCH_SIZE_CFD,
    "noise_mismatch_db_list": NOISE_MISMATCH_DB_LIST,
    "noise_sensitivity_snr_db": NOISE_SENSITIVITY_SNR_DB,
    "roc_snr_points": ROC_SNR_POINTS,
    "roc_num_thresholds": ROC_NUM_THRESHOLDS,
    "snapshot_points": SNAPSHOT_POINTS,
    "snapshot_snr_db": SNAPSHOT_SNR_DB,
    "cfd_alpha_grid": CFD_ALPHA_GRID,
    "cfd_tau_grid": CFD_TAU_GRID,
}

t0 = time.perf_counter()
performance_data, evaluation_summary = run_evaluation(eval_config)
t1 = time.perf_counter()

comparison_table = [
    {
        "method": "Proposed (Closed-form NP theory)",
        "pd_-10dB": performance_data["metric_curve"]["proposed"][SNR_POINTS_DB.index(-10)],
    },
    {
        "method": "NP analytic threshold (MC)",
        "pd_-10dB": performance_data["metric_curve"]["np_analytic"][SNR_POINTS_DB.index(-10)],
        "pfa_mean": float(np.mean(performance_data["pfa_curve"]["np_analytic"])),
    },
    {
        "method": "MC-CFAR threshold (MC)",
        "pd_-10dB": performance_data["metric_curve"]["cfar_mc"][SNR_POINTS_DB.index(-10)],
        "pfa_mean": float(np.mean(performance_data["pfa_curve"]["cfar_mc"])),
    },
    {
        "method": "NP-optimal benchmark",
        "pd_-10dB": performance_data["metric_curve"]["np_optimal"][SNR_POINTS_DB.index(-10)],
    },
    {
        "method": "Cyclostationary baseline",
        "pd_-10dB": performance_data["metric_curve"]["cyclostationary"][SNR_POINTS_DB.index(-10)],
        "pfa_mean": float(np.mean(performance_data["pfa_curve"]["cyclostationary"])),
    },
]

report_assets = {
    "problem_summary": "NP energy detector for AWGN SISO sensing with N=256 and Pfa<=0.01.",
    "solution_summary": "Closed-form NP threshold/theory plus vectorized Monte Carlo validation, CFAR and CFD baselines, ROC/AUC, and robustness checks.",
    "evaluation_summary": evaluation_summary,
    "tables": comparison_table,
    "figures": []
}

RESULTS = {
    "algorithm": "Closed-Form NP Energy Detector with Vectorized Monte Carlo Validation",
    "elapsed_sec": float(t1 - t0),
    "performance_data": performance_data,
    "report_assets": report_assets
}

# JSON-serializability sanity check
_ = json.dumps(RESULTS)

# AUTO-WISP EXECUTION CONTRACT
if not isinstance(globals().get('EVAL_CONFIG'), dict):
    EVAL_CONFIG = {}
if not isinstance(globals().get('RESULTS'), dict):
    RESULTS = run_experiment(globals().get('EVAL_CONFIG', {}))
RESULTS = _autowisp_normalize_results(RESULTS)
RESULTS.setdefault('notes', {})['primary_metric'] = 'P_d_at_P_fa_0_01'



Finally we create required figures and store filenames in `RESULTS["report_assets"]["figures"]`.

- **Fig 1:** $P_d$ vs SNR for all methods (primary metric).
- **Fig 2:** Snapshot sensitivity ($N$ vs $P_d$) at fixed SNR.
- **Fig 3:** ROC curves at selected SNR and AUC annotations.
- Saved with `fig.savefig(..., dpi=150, bbox_inches='tight')`.

In [ ]:
perf = RESULTS["performance_data"]

# Fig 1: Primary metric vs SNR (higher is better => linear scale)
fig1, ax1 = plt.subplots(figsize=(7, 4.5))
x = perf["metric_curve"]["x"]
ax1.plot(x, perf["metric_curve"]["proposed"], "k-", lw=2, label="Proposed (theory)")
ax1.plot(x, perf["metric_curve"]["np_analytic"], "o-", label="NP analytic (MC)")
ax1.plot(x, perf["metric_curve"]["cfar_mc"], "s-", label="CFAR (MC)")
ax1.plot(x, perf["metric_curve"]["np_optimal"], "k--", lw=1.5, label="NP-optimal benchmark")
ax1.plot(x, perf["metric_curve"]["cyclostationary"], "d-", label="Cyclostationary")
ax1.set_xlabel(perf["metric_curve"]["x_label"])
ax1.set_ylabel(perf["metric_curve"]["y_label"])
ax1.set_title("Fig 1: Detection Probability vs SNR")
ax1.grid(True, ls="--", alpha=0.4)
ax1.legend(loc="lower right", fontsize=8)
fig1_name = "fig1_pd_vs_snr.png"
fig1.savefig(fig1_name, dpi=150, bbox_inches="tight")

# Fig 2: Snapshot sensitivity
fig2, ax2 = plt.subplots(figsize=(7, 4.5))
xs = perf["snapshot_sensitivity"]["x"]
ax2.plot(xs, perf["snapshot_sensitivity"]["proposed"], "k-", lw=2, label="Proposed (theory)")
ax2.plot(xs, perf["snapshot_sensitivity"]["np_analytic"], "o-", label="NP analytic (MC)")
ax2.plot(xs, perf["snapshot_sensitivity"]["cfar_mc"], "s-", label="CFAR (MC)")
ax2.plot(xs, perf["snapshot_sensitivity"]["cyclostationary"], "d-", label="Cyclostationary")
ax2.set_xlabel(perf["snapshot_sensitivity"]["x_label"])
ax2.set_ylabel("Pd @ Pfa=0.01, SNR=-10 dB")
ax2.set_title("Fig 2: Snapshot Sensitivity")
ax2.grid(True, ls="--", alpha=0.4)
ax2.legend(fontsize=8)
fig2_name = "fig2_snapshot_sensitivity.png"
fig2.savefig(fig2_name, dpi=150, bbox_inches="tight")

# Fig 3: ROC curves (algorithm-specific visualization)
fig3, ax3 = plt.subplots(figsize=(7, 4.5))
for snr_key, roc_obj in perf["roc_curves"].items():
    auc_val = perf["roc_auc"][snr_key]
    ax3.plot(roc_obj["pfa"], roc_obj["pd"], lw=1.8, label=f"SNR={snr_key} dB, AUC={auc_val:.3f}")
ax3.plot([0, 1], [0, 1], "k--", alpha=0.6)
ax3.set_xlabel("Pfa")
ax3.set_ylabel("Pd")
ax3.set_title("Fig 3: ROC Curves (NP Energy Detector)")
ax3.grid(True, ls="--", alpha=0.4)
ax3.legend(fontsize=8, loc="lower right")
fig3_name = "fig3_roc_curves.png"
fig3.savefig(fig3_name, dpi=150, bbox_inches="tight")

RESULTS["report_assets"]["figures"] = [
    {"file": fig1_name, "caption": "Primary Pd-vs-SNR comparison"},
    {"file": fig2_name, "caption": "Sensitivity to number of samples N"},
    {"file": fig3_name, "caption": "ROC curves and AUC at selected SNRs"},
]

### Result Notes

The implementation satisfies the NP sensing workflow with closed-form thresholding and Monte Carlo cross-validation under AWGN.  
Primary expectation check: as SNR increases, all $P_d$ curves improve.

| Method | Threshold Design | Uses Signal Structure | Complexity per Interval | Role in Study |
|---|---|---:|---:|---|
| Proposed (Closed-form NP theory) | $\eta=\frac12\chi^{-1}_{2N}(1-\alpha)$ | No | $\mathcal{O}(N)$ | Reference analytic curve |
| NP analytic threshold (MC) | Same as NP theory | No | $\mathcal{O}(N)$ | Empirical validator |
| MC-CFAR energy | Quantile on noise-only $U_0$ | No | $\mathcal{O}(N)$ + calibration | Consistency baseline |
| NP-optimal benchmark | NP lemma equivalent | No | $\mathcal{O}(N)$ | Optimality reference |
| Cyclostationary baseline | Quantile on $T_{\mathrm{CFD}}$ | Yes (cyclic features) | Higher than $\mathcal{O}(N)$ | Robustness/complexity reference |

### Key operating-point comparison (from executed run metadata)

| Operating Point | Quantity | Where to Read |
|---|---|---|
| SNR = -10 dB | $P_d$ (proposed / NP / CFAR / CFD) | `RESULTS["performance_data"]["metric_curve"]` |
| All SNR points | $|\widehat{P}_d - P_d^{th}|$ | `RESULTS["performance_data"]["theory_empirical_gap"]` |
| Constraint check | empirical $P_{fa}$ curves | `RESULTS["performance_data"]["pfa_curve"]` |
| ROC/AUC | selected SNRs | `RESULTS["performance_data"]["roc_curves"]`, `["roc_auc"]` |
| Noise uncertainty | mismatch $\{-1,0,1\}$ dB | `RESULTS["performance_data"]["noise_uncertainty_sensitivity"]` |

The full structured output is available in `RESULTS` with required keys:
`algorithm`, `elapsed_sec`, `performance_data`, and `report_assets`.

# detection

This notebook implements the **Notebook Solution** solution as a single executable workflow.